# Predictive Information Learning Curves (N3)

This notebook demonstrates how to choose a sufficient history length using the predictive information learning-curve service.

It covers:
- short-memory vs persistent-memory behavior,
- plateau detection and recommended lookback,
- reliability caveats for finite samples and high-dimensional history windows.

In [ ]:
%matplotlib inline
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from forecastability.utils.datasets import generate_ar1, load_air_passengers
from forecastability.services.predictive_info_learning_curve_service import (
    build_predictive_info_learning_curve,
)

os.environ.setdefault("MPLBACKEND", "inline")
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print("Environment ready")

In [ ]:
series_map: dict[str, np.ndarray] = {
    "ar1_phi_0_8_n500": generate_ar1(n_samples=500, phi=0.8, random_state=42),
    "ar1_phi_0_97_n1200": generate_ar1(n_samples=1200, phi=0.97, random_state=7),
    "air_passengers_n144": load_air_passengers(),
}

for name, series in series_map.items():
    print(f"{name}: n={series.size}")

In [ ]:
curves = {
    name: build_predictive_info_learning_curve(
        series,
        max_k=8,
        random_state=42 + idx,
    )
    for idx, (name, series) in enumerate(series_map.items())
}

summary_rows: list[dict[str, object]] = []
for name, curve in curves.items():
    summary_rows.append(
        {
            "series": name,
            "plateau_detected": curve.plateau_detected,
            "recommended_lookback": curve.recommended_lookback,
            "convergence_index": curve.convergence_index,
            "max_information": float(np.max(curve.information_values))
            if curve.information_values
            else np.nan,
            "n_warnings": len(curve.reliability_warnings),
        }
    )

summary_df = pd.DataFrame(summary_rows).sort_values("series")
summary_df

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, (name, curve) in zip(axes, curves.items(), strict=True):
    k_vals = np.asarray(curve.window_sizes, dtype=int)
    i_vals = np.asarray(curve.information_values, dtype=float)

    ax.plot(k_vals, i_vals, marker="o", lw=1.8, color="steelblue", label="EvoRate(k)")
    if curve.plateau_detected:
        ax.axvline(
            curve.recommended_lookback,
            color="crimson",
            ls="--",
            lw=1.3,
            label="recommended lookback",
        )

    ax.set_title(name)
    ax.set_xlabel("Lookback k")
    ax.grid(alpha=0.25)

axes[0].set_ylabel("Predictive information (nats)")
axes[0].legend(fontsize=8)
fig.suptitle("Predictive Information Learning Curves", fontsize=12, fontweight="bold")
fig.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, (name, curve) in zip(axes, curves.items(), strict=True):
    k_vals = np.asarray(curve.window_sizes, dtype=int)
    i_vals = np.asarray(curve.information_values, dtype=float)
    gains = np.diff(i_vals, prepend=i_vals[0])

    ax.bar(k_vals, gains, color="darkorange", alpha=0.85)
    ax.axhline(0.0, color="black", lw=0.8)
    if curve.plateau_detected:
        ax.axvline(curve.recommended_lookback, color="crimson", ls="--", lw=1.2)
    ax.set_title(f"marginal gains: {name}")
    ax.set_xlabel("Lookback k")
    ax.grid(alpha=0.25)

axes[0].set_ylabel("Incremental gain in predictive information")
fig.tight_layout()
plt.show()

## Reliability Caveats

- The service caps lookback at `k=8` to limit curse-of-dimensionality effects.
- Shorter series can produce unstable kNN estimates for larger history windows.
- A detected plateau is a practical stopping rule, not a proof of true finite memory.
- Recommended lookback should be combined with rolling-origin model validation.

In [ ]:
for name, curve in curves.items():
    print(f"\n{name}")
    print(f"  plateau_detected: {curve.plateau_detected}")
    print(f"  recommended_lookback: {curve.recommended_lookback}")
    if curve.reliability_warnings:
        for warning in curve.reliability_warnings:
            print(f"  warning: {warning}")
    else:
        print("  warning: none")